# KNN (K Nearest Neighbors)

In [ ]:
'''
- Lazy Algorithm
- Find Number of Optimal Neighbors (K)
    -> Elbow Method
        -> Error Rate (Lowest Point) or Accuracy Score (Highest Point)

        -> Calculate Error Rate
            -> Total Wrong Prediction / Total Test Data

model.fit(X_train, Y_train)
model.predict(X_test)

100 Records

80% Train
20% Test

Email Spam -> 0 (Not Spam), 1 (Spam)

k_range = range(1, 21)

best range (3 to 7)
considerable range (12)

if error_rate or acc_rate
    - less than 3 -> Model Overfit
    - more than 12 -> Model underfit

Grid Search Cross Validation

range 1 -> error_rate
range 2 -> error_rate
range 3 -> error_rate
...
range 20 -> error_rate


k_range = range(1, 21)
for i in k_range:
    knn = KNearestClassifier(n_neighbors=i)
    knn.fit(X_train, Y_train)
    knn_pred = knn.predict(X_test)
    error_rate.append(np.mean(Y_test != knn_pred))
    data.append(acc_score(Y_test, Y_pred))
'''

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import KNN Model
from sklearn.neighbors import KNeighborsClassifier
# Import Train Test Split
from sklearn.model_selection import train_test_split
# Import Data Preprocessing Modules
from sklearn.preprocessing import StandardScaler, LabelEncoder
# Import Metrics
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [5]:
df = pd.read_csv('Dataset/student.csv')
df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,higher education,higher education,at_home,teacher,...,very good,medium,high,very low,very low,good,6,5,6,6
1,GP,F,17,U,GT3,T,primary education (4th grade),primary education (4th grade),at_home,other,...,excellent,medium,medium,very low,very low,good,4,5,5,6
2,GP,F,15,U,LE3,T,primary education (4th grade),primary education (4th grade),at_home,other,...,very good,medium,low,low,medium,good,10,7,8,10
3,GP,F,15,U,GT3,T,higher education,5th to 9th grade,health,services,...,good,low,low,very low,very low,excellent,2,15,14,15
4,GP,F,16,U,GT3,T,secondary education,secondary education,other,other,...,very good,medium,low,very low,low,excellent,4,6,10,10


In [9]:
## Continuous data to Category
df['G3'].sort_values(ascending=True).unique()

array([ 0,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
       20])

In [11]:
def convertMarks(marks):
    if marks >= 15:
        return 'A'
    elif marks >= 11:
        return 'B'
    elif marks >= 7:
        return 'C'
    else:
        return 'Fail'

In [12]:
df['Grade'] = df['G3'].apply(convertMarks)

In [14]:
df.columns

Index(['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
       'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime',
       'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences', 'G1', 'G2', 'G3', 'Grade'],
      dtype='object')

In [ ]:
columns = 'Medu', 'Fedu', 'traveltime', 'studytime',
       'failures', 'schoolsup', 'famsup','nursery','internet', 'freetime', 'goout', 'Dalc','Walc',
        'absences', 'G1', 'G2', 'Grade'

le_columns = 'schoolsup', 'famsup','nursery','internet'

## Feature Encoding

In [20]:
# Label Encoder
le = LabelEncoder()

def encodeData(dataframe, *args):
    for col in args:
        dataframe[f'{col}_id'] = le.fit_transform(dataframe[col])
    print('Columns Encoded.')

encodeData(df, 'schoolsup', 'famsup','nursery','internet')

Columns Encoded.


In [19]:
# val = ['C', 'A', 'B', 'D']

# data = le.fit_transform(val)
# data
# inv_data = le.inverse_transform(data)
# inv_data

In [56]:
df['health'].unique()

array(['good', 'excellent', 'very bad', 'bad', 'very good'], dtype=object)

In [24]:
medu_id = {
    'none': 0,
    'primary education (4th grade)': 1,
    '5th to 9th grade': 2,
    'secondary education': 3,
    'higher education': 4
}
df['medu_id'] = df['Medu'].map(medu_id)

In [37]:
# Manual Encoding
def manualEncode(dataframe, col, *args):
    encode_val = {}
    for idx, val in enumerate(args):
        encode_val[val] = idx

    dataframe[f'{col}_id'] = dataframe[col].map(encode_val)

In [38]:
manualEncode(df, 'Fedu', 'none', 'primary education (4th grade)', '5th to 9th grade','secondary education',
        'higher education')

In [41]:
manualEncode(df, 'traveltime', '<15 min.',  '15 to 30 min.', '30 min. to 1 hour', '>1 hour')

In [43]:
manualEncode(df, 'studytime', '<2 hours', '2 to 5 hours', '5 to 10 hours', '>10 hours')

In [45]:
manualEncode(df, 'freetime', 'very low', 'low', 'medium', 'high', 'very high')
manualEncode(df, 'goout', 'very low', 'low', 'medium', 'high', 'very high')
manualEncode(df, 'Dalc', 'very low', 'low', 'medium', 'high', 'very high')
manualEncode(df, 'Walc', 'very low', 'low', 'medium', 'high', 'very high')

In [57]:
manualEncode(df, 'health', 'very bad', 'bad', 'good', 'very good', 'excellent')

In [48]:
manualEncode(df, 'Grade', 'Fail', 'C', 'B', 'A')

## Features and Target

In [61]:
features = ['medu_id', 'Fedu_id', 'traveltime_id', 'studytime_id', 'failures', 'schoolsup_id', 
             'famsup_id', 'nursery_id', 'internet_id', 'freetime_id', 'goout_id', 'Dalc_id',
             'Walc_id', 'health_id', 'absences', 'G1', 'G2']
target = 'Grade_id'

In [62]:
X = df[features]
Y = df[target]

## Train Test

In [63]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

## Feature Scaling

In [64]:
scaler = StandardScaler()
X_train_scale = scaler.fit_transform(X_train)
X_test_scale = scaler.transform(X_test)